# Clase 10 — CNN 3D e imagen médica (parte 2: el modelo)

**Objetivos de hoy**
- Construir una CNN tridimensional.
- Entrenarla sobre los volúmenes preparados en la clase anterior.
- Evaluar el resultado con la métrica AUC-ROC de la competición de Kaggle.

Consulta el [capítulo 8 del manual](../manual/08_cnn_3d_imagen_medica.md) para la explicación completa.

**Nota**: si no ejecutaste la Clase 9 en esta misma sesión, la primera celda regenera el dataset sintético desde cero.

In [ ]:
import numpy as np
import tensorflow as tf
import os

if os.path.exists("X_cnn3d.npy"):
    X = np.load("X_cnn3d.npy")
    y = np.load("y_cnn3d.npy")
else:
    def generar_volumen_sintetico(num_cortes=40, alto=128, ancho=128, con_nodulo=True, semilla=None):
        rng = np.random.default_rng(semilla)
        volumen = rng.normal(loc=0.2, scale=0.05, size=(num_cortes, alto, ancho)).astype("float32")
        if con_nodulo:
            centro = (num_cortes // 2, alto // 2, ancho // 2)
            radio = 8
            z, yy, xx = np.ogrid[:num_cortes, :alto, :ancho]
            mascara = (z - centro[0]) ** 2 + (yy - centro[1]) ** 2 + (xx - centro[2]) ** 2 <= radio ** 2
            volumen[mascara] += 0.6
        return np.clip(volumen, 0, 1)

    def redimensionar_volumen(volumen, tamano_destino=(64, 64, 32)):
        volumen = tf.expand_dims(volumen, axis=-1)
        volumen = tf.image.resize(volumen, tamano_destino[:2])
        volumen = tf.transpose(volumen, perm=[2, 0, 1, 3])
        volumen = tf.image.resize(volumen, [tamano_destino[2], tamano_destino[0]])
        volumen = tf.transpose(volumen, perm=[1, 2, 0, 3])
        return volumen

    volumenes, etiquetas = [], []
    for i in range(60):
        con_nodulo = i % 2 == 0
        v = redimensionar_volumen(generar_volumen_sintetico(con_nodulo=con_nodulo, semilla=i))
        volumenes.append(v.numpy())
        etiquetas.append(1 if con_nodulo else 0)
    X, y = np.array(volumenes), np.array(etiquetas)

print("Forma de X:", X.shape)

## 1. Divide en entrenamiento y test

In [ ]:
indices = np.arange(len(X))
np.random.seed(0)
np.random.shuffle(indices)
X, y = X[indices], y[indices]

corte = int(len(X) * 0.8)
X_entreno, X_test = X[:corte], X[corte:]
y_entreno, y_test = y[:corte], y[corte:]
print("Pacientes de entrenamiento:", len(X_entreno), "| Pacientes de test:", len(X_test))

## 2. Define la CNN 3D

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 32, 1)),
    tf.keras.layers.Conv3D(16, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling3D(pool_size=2),
    tf.keras.layers.Conv3D(32, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling3D(pool_size=2),
    tf.keras.layers.GlobalAveragePooling3D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer="adam",
               loss="binary_crossentropy",
               metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

modelo.summary()

## 3. Entrena y evalúa

In [ ]:
modelo.fit(X_entreno, y_entreno, epochs=8, batch_size=4, validation_split=0.2)

In [ ]:
resultados = modelo.evaluate(X_test, y_test)
print(dict(zip(modelo.metrics_names, resultados)))

## Resumen

- `Conv3D` y `MaxPooling3D` son las versiones tridimensionales de las capas que ya conocías del capítulo 4.
- AUC mide qué tan bien distingue el modelo entre pacientes con y sin nódulo, la métrica real de la competición de Kaggle.
- Con datos sintéticos —un patrón claro y fácil de aprender— el modelo debería acercarse a un AUC alto rápidamente; con datos reales, el reto es mucho mayor.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase10_practica_cnn3d_modelo.ipynb).